# L15 · Agentic RL: Learning Across Tool-Using Turns

## Goal

- distinguish a single response from a step MDP
- mask tool output from policy loss
- compare outcome and process credit

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L15:toy:42").hexdigest()
print(f"lesson=L15 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L15 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:6991fdc13d01bf81da402c446d2cd63648a4265912685088c353e1e91edc7ebb data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: LLM policy → **Agentic RL multi-turn MDP** → trajectory evaluation

$$G_t=r_t^{process}+\sum_{k=t}^{T}\gamma^{k-t}r_k^{outcome}$$

An agent emits a `CALL` or `FINAL` action from an observation and receives tool output as the next observation. Only policy-generated action tokens enter the loss; tool output is context. Outcome broadcast and discounted process return make different credit assumptions.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** When tool-output text enters the next context, should those tokens enter the current policy-loss mask? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>No. Training environment-generated tokens as policy actions breaks gradient ownership.</details>

In [2]:
from rl_study.agentic.envs import CalculatorToolEnv
from rl_study.agentic.trajectory import rollout_episode, update_policy
from rl_study.models import TinyCausalLM, TinyLMConfig, TinyTokenizer
agent_model = TinyCausalLM(TinyLMConfig(
    max_sequence_length=192, hidden_size=32, num_heads=4,
    num_layers=1, intermediate_size=64
))
agent_tokenizer = TinyTokenizer()
agent_trajectory, generated, rollout_forwards = rollout_episode(
    agent_model, agent_tokenizer, CalculatorToolEnv(seed=42, max_steps=3),
    generator=torch.Generator().manual_seed(42), policy_version=0, task_index=0
)
agent_optimizer = torch.optim.AdamW(agent_model.parameters(), lr=1e-3)
agent_update = update_policy(
    agent_model, agent_tokenizer, agent_optimizer, agent_trajectory,
    current_policy_version=0, credit_mode="discounted_returns", gamma=0.95
)
print({"episode_steps": len(agent_trajectory.steps), "generated_tokens": generated,
       "rollout_forwards": rollout_forwards, "credits": agent_update.step_credits,
       "tool_outputs_masked": True, "loss": round(agent_update.loss, 4)})

{'episode_steps': 1, 'generated_tokens': 14, 'rollout_forwards': 1, 'credits': (-0.25,), 'tool_outputs_masked': True, 'loss': -0.2648}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** The trajectory preserves rollout-time token IDs, candidate set, and policy version to detect retokenization and stale-policy drift. Text-only logs are readable but insufficient for exact updates.

**Common trap:** Storing outcome reward at every step and adding it again in discounted returns double-counts it. Store outcome once at termination and let the credit function distribute it. Regression tests: `test_rollout_preserves_original_action_tokens_and_masks_tool_output`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert len(agent_trajectory.steps) >= 1
assert all(step.action_token_ids for step in agent_trajectory.steps)
print("checks=passed")

checks=passed


**Recall:** What diagnostic ability is lost when process and outcome rewards are pre-merged into one scalar? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** This seeded rollout terminated after one step with credit -0.25 and a finite update loss. A failed rollout still provides valid evidence for mask and credit contracts.
- Executable checks: `test_rollout_preserves_original_action_tokens_and_masks_tool_output`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L16 audits reward hacking, budgets, splits, and initial hashes rather than looking at success alone.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

[Implementation note](../../docs/algorithms/agentic-rl.md) · [Course map](../../docs/course-map.en.md)

## Sources

- `agent-lightning-2025` — `docs/sources.yml`
- `agent-r1-2025` — `docs/sources.yml`
- `framework-agent-lightning` — `docs/sources.yml`
- `repo-agent-r1` — `docs/sources.yml`
- `benchmark-alfworld` — `docs/sources.yml`